In [1]:
%config Completer.use_jedi = False

# Imports
import laspy
import numpy as np
import time

# Decimal tolerance
EPS=1e-10
# Default k parameter
DEFAULT_k=1
# Steps for linearly spaced methods
STEPS=100
# Min points for the generated point cloud
MIN_POINTS=16000

# Variable definition
u, v = var('u, v')

In [2]:
# ---  ONE SHEET HYPERBOLOID  --- #
# ------------------------------- #
def parametric_onesheet_hyperboloid_init(k, a=1, b=1, c=1.5):
    urange = [0, 2*pi-EPS]
    vrange = [-1*k, 1*k]
    params = [k*a, k*b, k*c]
    return urange, vrange, params
    
def parametric_onesheet_hyperboloid(u, v, params):
    a, b, c = params
    return vector([
        a*cosh(v)*cos(u),
        b*cosh(v)*sin(u),
        c*sinh(v)
    ])
    
# ---  SPHERE  --- #
# ---------------- #
def parametric_sphere_init(k):
    urange = [EPS, pi]
    vrange = [EPS, 2*pi-EPS]
    params = [k]
    return urange, vrange, params
    
def parametric_sphere(u, v, params):
    r = params[0]
    return vector([
        r*sin(u)*cos(v),
        r*sin(u)*sin(v),
        r*cos(u)
    ])
    
# ---  ELLIPSOID  --- #
# ------------------- #
def parametric_ellipsoid_init(k, a=0.9, b=1.1, c=1.6):
    urange = [EPS, pi]
    vrange = [EPS, 2*pi-EPS]
    params = [k*a, k*b, k*c]
    return urange, vrange, params

def parametric_ellipsoid(u, v, params):
    a, b, c = params
    return vector([
        a*sin(u)*cos(v),
        b*sin(u)*sin(v),
        c*cos(u)
    ])
    
# ---  ELLIPTICAL CYLINDER  --- #
# ----------------------------- #
def parametric_elliptical_cylinder_init(h, a=0.6, b=0.5):
    urange = [-h, h]
    vrange = [0, 2*pi-EPS]
    params = [a, b]
    return urange, vrange, params
    
def parametric_elliptical_cylinder(u, v, params):
    a, b = params
    return vector([
        a*cos(v),
        b*sin(v),
        u
    ])
    
# ---  HYPERBOLIC PARABOLOID  --- #
# ------------------------------- #
def parametric_hyperbolic_paraboloid_init(k, a=0.6, b=0.9):
    urange = [-k, k]
    vrange = [-k, k]
    params = [a, b]
    return urange, vrange, params

def parametric_hyperbolic_paraboloid(u, v, params):
    a, b = params
    return vector([
        a*u,
        b*v,
        b**2*v**2 - a**2*u**2
    ])
    
# ---  TORUS  --- #
# --------------- #
def parametric_torus_init(k, r=0.5, R=1.0):
    urange = [0, 2*pi-EPS]
    vrange = [0, 2*pi-EPS]
    params = [k*r, k*R]
    return urange, vrange, params

def parametric_torus(u, v, params):
    r, R = params
    return vector([
        cos(u)*(R+r*cos(v)),
        sin(u)*(R+r*cos(v)),
        r*sin(v)
    ])
    
# ---   CATENOID   --- #
# -------------------- #
def parametric_catenoid_init(h, a=1.3):
    urange = [-h, h]
    vrange = [0, 2*pi-EPS]
    params = [a]
    return urange, vrange, params

def parametric_catenoid(u, v, params):
    a = params[0]
    return vector([
        a*cosh(u/a)*cos(v),
        a*cosh(u/a)*sin(v),
        u
    ])
    
# ---  SELECT THE MODEL  --- #
# -------------------------- #
initializer = parametric_torus_init
urange, vrange, params = initializer(DEFAULT_k)
g = parametric_torus(u, v, params)


# ---  PLOT THE MODEL  --- #
# ------------------------ #
plot = parametric_plot3d(g, urange, vrange)
plot.show()

Graphics3d Object

### How to compute curvature-based geometric features

Let $\pmb{g}(u, v) = (x(u, v), y(u, v), z(u, v))$ be a parametric surface model. Note that the orthonormal vector at any point of this surface is given by $\pmb{n} = \left\lVert\dfrac{\partial \pmb{g}}{\partial u} \times \dfrac{\partial \pmb{g}}{\partial v}\right\rVert^{-1} \left(\dfrac{\partial \pmb{g}}{\partial u} \times \dfrac{\partial \pmb{g}}{\partial v}\right)$.

#### 1) First fundamental form $\pmb{F}_1$

$$
\pmb{F}_1 = \left[\begin{array}{cc}
    \left\langle{\dfrac{\partial \pmb{g}}{\partial u}, \dfrac{\partial \pmb{g}}{\partial u}}\right\rangle &
        \left\langle{\dfrac{\partial \pmb{g}}{\partial u}, \dfrac{\partial \pmb{g}}{\partial v}}\right\rangle \\
    \left\langle{\dfrac{\partial \pmb{g}}{\partial u}, \dfrac{\partial \pmb{g}}{\partial v}}\right\rangle &
        \left\langle{\dfrac{\partial \pmb{g}}{\partial v}, \dfrac{\partial \pmb{g}}{\partial v}}\right\rangle
\end{array}\right]
$$

#### 2) Second fundamental form $\pmb{F}_2$

$$
\pmb{F}_2 = \left[\begin{array}{cc}
    \left\langle{\dfrac{\partial^2 \pmb{g}}{\partial u^2}, \pmb{n}}\right\rangle &&
        \left\langle{\dfrac{\partial^2 \pmb{g}}{\partial u \partial v}, \pmb{n}}\right\rangle \\
    \left\langle{\dfrac{\partial^2 \pmb{g}}{\partial u \partial v}, \pmb{n}}\right\rangle &&
        \left\langle{\dfrac{\partial^2 \pmb{g}}{\partial v^2}, \pmb{n}}\right\rangle
\end{array}\right]
$$

#### 3) Principal curvatures $\kappa_1, \kappa_2$

The principal curvatures are given by the eigenvalues of $\pmb{F}_1^{-1}\pmb{F}_2$.

#### 4) Gaussian curvature

$$
K = \kappa_1 \kappa_2
$$

#### 5) Mean curvature

$$
H = \dfrac{\kappa_1 + \kappa_2}{2}
$$

#### 6) Shape index

$$
S = -\dfrac{2}{\pi} \arctan\left(\dfrac{H}{\sqrt{H^2-K}}\right)
$$

#### 7) Umbilical deviation

$$
u_{\text{dev}} = \lvert\kappa_1 - \kappa_2\rvert
$$

#### 8) Root mean squared curvature

$$
\text{RMSC} = \sqrt{\dfrac{\kappa_1^2+\kappa_2^2}{2}}
$$

#### 9) Umbilicality

$$
u_\text{exp} = \exp\left(-\lvert\kappa_1-\kappa_2\rvert\right)
$$

#### 10) Gaussian umbilicality

$$
u_\text{gauss} = \exp\left[-(\kappa_1-\kappa_2)^2\right]
$$


In [3]:
# ---  COMPUTE THE MODEL  --- #
# --------------------------- #
# Compute first and second partial derivatives
dgdu = diff(g, u)
dgdv = diff(g, v)
d2gdu2 = diff(dgdu, u)
d2gdudv = diff(dgdu, v)
d2gdv2 = diff(dgdv, v)
# Compute surface normal
N = dgdu.cross_product(dgdv)
N = N / sqrt(N.dot_product(N))
# Compute first fundamental form
F1 = matrix([
    [dgdu.dot_product(dgdu), dgdu.dot_product(dgdv)],
    [dgdu.dot_product(dgdv), dgdv.dot_product(dgdv)]
])
# Compute second fundamental form
F2 = matrix([
    [d2gdu2.dot_product(N), d2gdudv.dot_product(N)],
    [d2gdudv.dot_product(N), d2gdv2.dot_product(N)]
])
# Compute principal curvatures
kappa1, kappa2 = (F1.inverse()*F2).eigenvalues()
# Compute Gaussian curvature
K = kappa1*kappa2
# Compute mean curvature
H = (kappa1+kappa2)/2
# Compute shape index
S = -2/pi * arctan(H/sqrt(H*H-K))
if initializer == parametric_sphere_init:  # Handle particular case of the sphere
    S = H-H+1
# Compute umbilical deviation
udev = abs(kappa1-kappa2)
# Compute root mean squared curvature
rmsc = sqrt((kappa1*kappa1+kappa2*kappa2)/2)
# Compute umbilicality
uexp = exp(-abs(kappa1-kappa2))
# Compute Gaussian umbilicality
ugauss = exp(-(kappa1-kappa2)^2)


# ---  PLOT THE MODEL  --- #
# ------------------------ #
Kmin = [
    K(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
    )
]
Kmin, Kmax = min(Kmin), max(Kmin)
def color_K(u, v):
    return (K(u=u, v=v).n()-Kmin)/(Kmax-Kmin)
Hmin = [
    H(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
    )
]
Hmin, Hmax = min(Hmin), max(Hmin)
def color_H(u, v):
    return (H(u=u, v=v).n()-Hmin)/(Hmax-Hmin)
Smin = [
    S(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
    )
]
Smin, Smax = min(Smin), max(Smin)
def color_S(u, v):
    return (S(u=u, v=v).n()-Smin)/(Smax-Smin) 
udevmin = [
    udev(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
    )
]
udevmin, udevmax = min(udevmin), max(udevmin)
def color_udev(u, v):
    return (udev(u=u, v=v).n()-udevmin)/(udevmax-udevmin) 
rmscmin = [
    rmsc(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
    )
]
rmscmin, rmscmax = min(rmscmin), max(rmscmin)
def color_rmsc(u, v):
    return (rmsc(u=u, v=v).n()-rmscmin)/(rmscmax-rmscmin) 
uexpmin = [
    uexp(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
    )
]
uexpmin, uexpmax = min(uexpmin), max(uexpmin)
def color_uexp(u, v):
    return (uexp(u=u, v=v).n()-uexpmin)/(uexpmax-uexpmin) 
ugaussmin = [
    ugauss(u=ui, v=vi).n() for (ui, vi) in zip(
        np.linspace(urange[0].n(), urange[1].n(), STEPS),
        np.linspace(vrange[0].n(), vrange[1].n(), STEPS)
)
]
ugaussmin, ugaussmax = min(ugaussmin), max(ugaussmin)
def color_ugauss(u, v):
    return (ugauss(u=u, v=v).n()-ugaussmin)/(ugaussmax-ugaussmin) 


urange, vrange, params = initializer(DEFAULT_k)
plot = parametric_plot3d(g, urange, vrange, color=(color_K, colormaps.jet))
plot.show()

Graphics3d Object

In [4]:
# Generate the point cloud
start = time.perf_counter()
X, F = [], []
steps = int(np.ceil(np.sqrt(MIN_POINTS)))
for ui in np.linspace(urange[0].n(), urange[1].n(), steps):
    for vi in np.linspace(vrange[0].n(), vrange[1].n(), steps):
        xi = g(u=ui, v=vi)
        X.append(xi)
        kappa1i = kappa1(u=ui, v=vi)
        kappa2i = kappa2(u=ui, v=vi)
        maxabscurv = np.max([np.abs(kappa1i), np.abs(kappa2i)])
        minabscurv = np.min([np.abs(kappa1i), np.abs(kappa2i)])
        Ki = K(u=ui, v=vi)
        Hi = H(u=ui, v=vi)
        Si = S(u=ui, v=vi)
        udevi = udev(u=ui, v=vi)
        rmsci = rmsc(u=ui, v=vi)
        uexpi = uexp(u=ui, v=vi)
        ugaussi = ugauss(u=ui, v=vi)
        F.append([kappa1i, kappa2i, maxabscurv, minabscurv, Ki, Hi, Si, udevi, rmsci, uexpi, ugaussi])
P = np.hstack([X, F])
end = time.perf_counter()
print(f'Generated {P.shape} point cloud in {end-start:.3f} seconds.')

Generated (16129, 14) point cloud in 93.602 seconds.


In [5]:
# Write generated point cloud
header = laspy.LasHeader(point_format=int(6), version="1.4")  # Header
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_pcurv1", type=np.float32))  # Extra attribute (kappa1)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_pcurv2", type=np.float32))  # Extra attribute (kappa2)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_maxabscurv", type=np.float32))  # Extra attribute (maxabscurv)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_minabscurv", type=np.float32))  # Extra attribute (minabscurv)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_gauss_curv", type=np.float32))  # Extra attribute (K)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_mean_curv", type=np.float32))  # Extra attribute (H)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_shape_index", type=np.float32))  # Extra attribute (S)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_umbildev", type=np.float32))  # Extra attribute (udev)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_RMSC", type=np.float32))  # Extra attribute (rmsc)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_umbilicality", type=np.float32))  # Extra attribute (uexp)
header.add_extra_dim(laspy.ExtraBytesParams(name="sage_gumbilicality", type=np.float32))  # Extra attribute (ugauss)
las = laspy.LasData(header)  # Build LAS from header
las.x = P[:, 0].astype(np.float32)  # Populate x coordinate
las.y = P[:, 1].astype(np.float32)  # Populate y coordinate
las.z = P[:, 2].astype(np.float32)  # Populate z coordinate
las.sage_pcurv1 = P[:, 3].astype(np.float32)  # Populate kappa1
las.sage_pcurv2 = P[:, 4].astype(np.float32)  # Populate kappa2
las.sage_maxabscurv = P[:, 5].astype(np.float32)  # Populate maxabscurv
las.sage_minabscurv = P[:, 6].astype(np.float32)  # Populate minabscurv
las.sage_gauss_curv = np.abs(P[:, 7].astype(np.float32))  # Populate K
las.sage_mean_curv = np.abs(P[:, 8].astype(np.float32))  # Populate H
las.sage_shape_index = np.abs(P[:, 9].astype(np.float32))  # Populate S
las.sage_umbildev = np.abs(P[:, 10].astype(np.float32))  # Populate udev
las.sage_RMSC = np.abs(P[:, 11].astype(np.float32))  # Populate rmsc
las.sage_umbilicality = np.abs(P[:, 12].astype(np.float32))  # Populate uexp
las.sage_gumbilicality = np.abs(P[:, 13].astype(np.float32))  # Populate ugauss
las.write("/ext4/lidar_data/math/geomfeats2nd/sage_tor_r0_5_R1_0.laz")  # Write to file